# Notebook 02 - Colour and grayscale

*Time: about 45 minutes*

## What you will learn
- A colour picture is **three grayscale pictures stacked**: blue, green and red channels.
- How to convert a picture to **grayscale** (one number per pixel).
- Simple **brightness** and **contrast** changes, and why `cv2.add` is safer than `+`.
- How to **resize** and **flip** a picture.

## Prerequisites
- Notebook 01 (shape, slicing, BGR order).

### What this code does

We load the libraries. This time we also use `numpy` (we call it `np`), the library that holds the grid of numbers, and two helpers from `utils.py`: `load` (reads a picture) and `show_many` (draws several pictures side by side).

In [ ]:
import sys                                    # lets us change where Python looks for files
sys.path.append("..")                         # also look one folder up (utils.py lives there)
import cv2                                    # OpenCV
import numpy as np                            # NumPy: maths on grids of numbers
from utils import load, show_many             # our helpers
print("Libraries loaded - ready to go!")      # a visible sign that everything worked

### What you should see / Why we care

`Libraries loaded - ready to go!`

### What this code does

We set `IMAGE_PATH` (the one line to change for another picture), load the picture and print its shape.

In [ ]:
IMAGE_PATH = "../assets/objects_shapes.png"   # <- change this line to use another picture
img = load(IMAGE_PATH)                        # read the picture (BGR order)
print("color image shape:", img.shape)        # 3 numbers per pixel

### What you should see / Why we care

`color image shape: (420, 600, 3)`

### What this code does

`cv2.split` cuts the colour picture into its three channels. Each channel is a **gray picture** (2D grid, one number per pixel). A bright pixel in the *red channel* means "lots of red here".

We show the three channels as three subplots. Remember the order: **blue, green, red**.

In [ ]:
blue, green, red = cv2.split(img)             # three separate 2D grids
print("one channel shape:", red.shape)        # only height and width now
show_many([blue, green, red],                 # the three pictures
          ["blue channel", "green channel", "red channel"],  # their titles
          columns=3, size=(12, 3))            # three side by side

### What you should see / Why we care

Each channel is a gray picture. In the **red channel** the red circles are **bright (white)**. In the **green channel** the green squares are bright, in the **blue channel** the blue triangles are bright. The white background is bright in all three because white = full red + full green + full blue.

### What this code does

Another way to *see* a single colour: make a black picture the same size, and copy only one channel into it. `np.zeros_like(img)` makes an all-zero (black) picture. Channel **0 = blue, 1 = green, 2 = red**.

In [ ]:
only_blue = np.zeros_like(img)                # black picture, same size as img
only_green = np.zeros_like(img)               # another black picture
only_red = np.zeros_like(img)                 # and another
only_blue[:, :, 0] = blue                     # fill channel 0 (blue)
only_green[:, :, 1] = green                   # fill channel 1 (green)
only_red[:, :, 2] = red                       # fill channel 2 (red)
show_many([only_blue, only_green, only_red],  # show the three results
          ["B only", "G only", "R only"], columns=3, size=(12, 3))

### What you should see / Why we care

Three pictures tinted blue, green and red. Notice the **white background turns into the pure colour**, and shapes of the *other* colours turn black. This is the foundation of colour masking in Notebook 07.

### What this code does

`cv2.cvtColor(..., cv2.COLOR_BGR2GRAY)` mixes the three channels into **one** number per pixel (it weights green most because our eyes are most sensitive to green). Gray pictures are smaller, simpler and perfect for thresholding.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # colour -> gray
print("gray shape:", gray.shape)              # no channel number any more
print("gray value of a red circle pixel  :", gray[90, 100])    # red is fairly dark
print("gray value of a green square pixel:", gray[210, 105])   # green is brighter
show_many([img, gray], ["color (3 numbers per pixel)", "gray (1 number per pixel)"],
          columns=2, size=(10, 4))            # compare

### What you should see / Why we care

`gray shape: (420, 600)`, a gray value of about **66** for the red circle and **100** for the green square. The gray picture looks like an old black-and-white photo.

### What this code does

**Brightness** means adding the same number to every pixel. We could write `img + 60`, but numbers above 255 would **wrap around** (250 + 10 becomes 4, a dark pixel!). `cv2.add` **stops at 255** instead (this is called *saturation*). First a tiny proof, then the real thing.

In [ ]:
a = np.array([[250]], dtype=np.uint8)         # one pixel with value 250
b = np.array([[10]], dtype=np.uint8)          # we want to add 10 to it
print("NumPy  250 + 10 =", a + b)             # wraps around: 4 (wrong!)
print("cv2.add 250 + 10 =", cv2.add(a, b))    # stops at 255 (right!)
brighter = cv2.add(img, np.full_like(img, 60))  # add 60 to every number
print("brightest value after adding 60:", brighter.max())  # never above 255

### What you should see / Why we care

NumPy gives `[[4]]` (wrong), `cv2.add` gives `[[255]]` (right). The brightest value stays `255`.

**Why we care:** image numbers live in the 0-255 box. Safe arithmetic keeps them inside it.

### What this code does

**Contrast** means *multiplying* the numbers (bright gets brighter, dark stays dark). `cv2.convertScaleAbs(img, alpha=1.5, beta=0)` does `pixel * 1.5 + 0` and keeps the result inside 0-255. `alpha` is the multiplier (contrast), `beta` is the number added (brightness).

In [ ]:
contrast = cv2.convertScaleAbs(img, alpha=1.5, beta=0)    # multiply by 1.5
darker = cv2.convertScaleAbs(img, alpha=0.5, beta=0)      # multiply by 0.5
show_many([img, brighter, contrast, darker],              # four versions
          ["original", "brighter (+60)", "contrast (x1.5)", "darker (x0.5)"],
          columns=4, size=(14, 3))                        # side by side

### What you should see / Why we care

The white background stays white (it cannot get brighter). Colours become lighter in "brighter", more intense in "contrast" and dimmer in "darker".

### What this code does

**Resizing.** `cv2.resize(img, (new_width, new_height))`. Careful: OpenCV wants **width first**, but `img.shape` gives **height first**! We make a half-size and a double-size copy.

In [ ]:
small = cv2.resize(img, (300, 210))           # width 300, height 210
big = cv2.resize(img, None, fx=2, fy=2)       # scale both sides by 2
print("original:", img.shape)                 # (420, 600, 3)
print("small   :", small.shape)               # (210, 300, 3)
print("big     :", big.shape)                 # (840, 1200, 3)

### What you should see / Why we care

`(420, 600, 3)`, `(210, 300, 3)` and `(840, 1200, 3)`. Notice the shapes print height first, even though we typed width first.

### What this code does

**Flipping.** `cv2.flip(img, 1)` mirrors left-right, `0` mirrors up-down and `-1` does both.

In [ ]:
flipped_lr = cv2.flip(img, 1)                 # left <-> right
flipped_ud = cv2.flip(img, 0)                 # up <-> down
flipped_both = cv2.flip(img, -1)              # both at once
show_many([img, flipped_lr, flipped_ud, flipped_both],   # four pictures
          ["original", "flip 1", "flip 0", "flip -1"], columns=4, size=(14, 3))

### What you should see / Why we care

The circles, squares and triangles change places. The triangles point **down** in the up-down flip.

## Recap
| Idea | Code |
|---|---|
| Split channels | `blue, green, red = cv2.split(img)` |
| Gray | `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` |
| Safe brightness | `cv2.add(img, np.full_like(img, 60))` |
| Contrast | `cv2.convertScaleAbs(img, alpha=1.5, beta=0)` |
| Resize (width, height!) | `cv2.resize(img, (300, 210))` |
| Flip | `cv2.flip(img, 1)` |

**Next:** Notebook 03 turns gray pictures into pure black and white (thresholding).